# 05 · Train / validation / test splits (CPU)

**What this does:** decides which buildings a model may learn from and which are held back for testing. This happens **before** any address text is written, so nothing from a test area can leak into training text.

**Two kinds of test data**

| Name | What the model has seen | What it measures |
|---|---|---|
| **warm** | other buildings in the same neighbourhoods | everyday accuracy |
| **cold** | nothing at all from that part of the city | can it handle a new area? (new launch zones, new housing) |

Cold areas are whole **H3 resolution-7 cells** (H3 is Uber's hexagon grid; a resolution-7 cell is about 5 sq km). About 15% of cells are held out, spread evenly between city centre and outskirts.

**One more held-back thing: hidden landmarks.** 20% of landmarks are marked "not public". They still appear in address text, because real people cite shops no map lists. A method that only looks names up cannot resolve them. This stops the benchmark from being a pure lookup exercise.

**Runtime:** under a minute, CPU. **Needs:** notebook 04.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/kaggle/working/Navigate-code' if os.path.exists('/kaggle/working') else '/content/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

In [ ]:
from bench import splits as sp
B = P['bench']
t  = pd.read_parquet(f'{B}/targets.parquet')
lm = pd.read_parquet(f'{B}/landmarks_clean.parquet')

t, cold_cells = sp.assign_splits(t, cold_frac=0.15, holdout_frac=0.08, val_frac=0.02, seed=7)
lm = sp.flag_landmarks(lm, cold_cells, hidden_frac=0.20, seed=7)

print(sp.split_report(t).to_string())
print(f"\n{len(cold_cells)} cold cells out of {t.h3_7.nunique()}")
print(f"landmarks: {len(lm):,} total | {(~lm.lm_public).mean():.1%} hidden from the gazetteer | {lm.in_cold_cell.mean():.1%} inside cold cells")
print("\nmedian distance to the nearest TRAINING building (metres):")
print(t.groupby('split').nn_train_m.median().round(0).to_string())

## Map of the split
Cold cells should look like scattered islands, not one corner of the city.

In [ ]:
import matplotlib.pyplot as plt
s = t.sample(min(40000, len(t)), random_state=0)
fig, ax = plt.subplots(figsize=(8, 7))
for name, col, size in [('train', '0.75', 1), ('val', '0.4', 2), ('holdout_warm', 'tab:blue', 3), ('holdout_cold', 'tab:red', 3)]:
    d = s[s.split == name]; ax.scatter(d.lon, d.lat, s=size, c=col, label=f'{name} ({(t.split == name).sum():,})')
ax.set_aspect('equal'); ax.legend(markerscale=4); ax.set_title('Target buildings by split'); plt.show()

In [ ]:
t.to_parquet(f'{B}/targets.parquet', index=False)                    # same file, now with split, ring, nn_train_m
lm.to_parquet(f'{B}/landmarks_clean.parquet', index=False)           # same file, now with lm_public, in_cold_cell
import json
json.dump({'cold_cells': cold_cells, 'seed': 7, 'cold_frac': 0.15, 'hidden_landmark_frac': 0.20}, open(f'{B}/splits.json', 'w'), indent=1)
print('saved')

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '05' + '_outputs', [f"{P['bench']}/{f}" for f in ('targets.parquet', 'landmarks_clean.parquet', 'splits.json')])